In [1]:
import sys
sys.version_info

sys.version_info(major=3, minor=6, micro=2, releaselevel='final', serial=0)

In [ ]:
def relu(x):
    if x > 0: return x
    else: return 0

In [6]:
from itertools import permutations, combinations, product

l = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']

In [8]:
res = []
for combination in [set(c) for c in permutations(l, 3)]:
    if combination not in res: res.append(combination)
res

[{'a', 'b', 'c'},
 {'a', 'b', 'd'},
 {'a', 'b', 'e'},
 {'a', 'b', 'f'},
 {'a', 'b', 'g'},
 {'a', 'b', 'h'},
 {'a', 'b', 'i'},
 {'a', 'b', 'j'},
 {'a', 'c', 'd'},
 {'a', 'c', 'e'},
 {'a', 'c', 'f'},
 {'a', 'c', 'g'},
 {'a', 'c', 'h'},
 {'a', 'c', 'i'},
 {'a', 'c', 'j'},
 {'a', 'd', 'e'},
 {'a', 'd', 'f'},
 {'a', 'd', 'g'},
 {'a', 'd', 'h'},
 {'a', 'd', 'i'},
 {'a', 'd', 'j'},
 {'a', 'e', 'f'},
 {'a', 'e', 'g'},
 {'a', 'e', 'h'},
 {'a', 'e', 'i'},
 {'a', 'e', 'j'},
 {'a', 'f', 'g'},
 {'a', 'f', 'h'},
 {'a', 'f', 'i'},
 {'a', 'f', 'j'},
 {'a', 'g', 'h'},
 {'a', 'g', 'i'},
 {'a', 'g', 'j'},
 {'a', 'h', 'i'},
 {'a', 'h', 'j'},
 {'a', 'i', 'j'},
 {'b', 'c', 'd'},
 {'b', 'c', 'e'},
 {'b', 'c', 'f'},
 {'b', 'c', 'g'},
 {'b', 'c', 'h'},
 {'b', 'c', 'i'},
 {'b', 'c', 'j'},
 {'b', 'd', 'e'},
 {'b', 'd', 'f'},
 {'b', 'd', 'g'},
 {'b', 'd', 'h'},
 {'b', 'd', 'i'},
 {'b', 'd', 'j'},
 {'b', 'e', 'f'},
 {'b', 'e', 'g'},
 {'b', 'e', 'h'},
 {'b', 'e', 'i'},
 {'b', 'e', 'j'},
 {'b', 'f', 'g'},
 {'b', 'f'

In [7]:
def get_k(n):
    k = 0
    while 2 ** k < n + k + 1:
        k += 1
    return k

def get_check_groups(n, k):
    check_groups = {}
    for i in range(1, k + 1):
        index = 2 ** (i - 1)
        check_groups[index] = []
        for j in range(index + 1, n + k + 1):
            if j & (1 << (i - 1)):
                check_groups[index].append(j)
    return check_groups

def get_check_code(group_data, parity):
    if parity == 'odd':
        return 1 - sum(group_data) % 2
    elif parity == 'even':
        return sum(group_data) % 2
    else:
        raise ValueError('parity must be odd or even')

class HammingCode:
    def __init__(self, data: (list, tuple, str, int)=None):
        if data is not None:
            self._process(data)

    def _process(self, data: (list, tuple, str, int)):
        if isinstance(data, str):
            data = [int(i) for i in data]
        elif isinstance(data, int):
            # 转为二进制列表
            data = [int(i) for i in bin(data)[2:]]
        self.data = data
        self.n = len(data)
        self.k = get_k(self.n)
        self.check_groups = get_check_groups(self.n, self.k)
    
    def _inverse_print(self, code):
        print("Hamming Code: {}".format("".join([str(i) for i in code[::-1]])))

    def _check_error(self, code=None):
        if code is None:
            code = self.code
        errors = []
        for i, group in self.check_groups.items():
            res = get_check_code([code[j - 1] for j in group + [i]], 'odd')
            print("check group: {}, check code: {}".format(group, res))
            errors.append(res)
        if sum(errors) == 0:
            return None
        else:
            return sum([2 ** (len(errors) - i - 1) for i, e in enumerate(errors) if e])

    def __call__(self, data=None):
        if data is not None:
            self._process(data)
        return self.encode()
    
    def encode(self):
        # 默认奇校验
        code = [0] * (self.n + self.k)
        data_index = 0
        for i in range(1, self.n + self.k + 1):
            if i in self.check_groups:
                code[i - 1] = None
            else:
                code[i - 1] = self.data[self.n - data_index - 1]
                data_index += 1
        for i, group in self.check_groups.items():
            code[i - 1] = get_check_code([code[j - 1] for j in group], 'even')
        # 倒序输出字符串
        self._inverse_print(code)
        self.code = code
        return code
    
    def decode(self, code=None):
        if code is None:
            code = self.code
        # 默认奇校验
        errors = self._check_error(code)
        if errors is not None:
            code[errors - 1] = 1 - code[errors - 1]
        data = []
        for i in range(1, self.n + self.k + 1):
            if i not in self.check_groups:
                data.append(code[i - 1])
        self.data = data[::-1]
        return "".join([str(i) for i in data[::-1]])

In [12]:
ham = HammingCode() # 从高位到低位，称之为大端序

In [13]:
hc = ham("111001")
hc

Hamming Code: 1101001111


[1, 1, 1, 1, 0, 0, 1, 0, 1, 1]

In [14]:
hc[1] = 0
hc

[1, 0, 1, 1, 0, 0, 1, 0, 1, 1]

In [15]:
ham._check_error(hc)

check group: [3, 5, 7, 9], check code: 1
check group: [3, 6, 7, 10], check code: 0
check group: [5, 6, 7], check code: 1
check group: [9, 10], check code: 1


13

In [6]:
ham._check_error2(hc)

AttributeError: 'HammingCode' object has no attribute '_check_error2'

In [1]:
# 信源
class Node:
    def __init__(self, symbol, prob):
        self.symbol = symbol
        self.prob = prob
        self.left = None
        self.right = None
        self.code = None

# 信源按照概率从小到大排序
def sort_nodes(nodes):
    return sorted(nodes, key=lambda x: x.prob)

# 合并两个信源
def merge_nodes(node1, node2):
    node = Node(None, node1.prob + node2.prob)
    node.left = node1
    node.right = node2
    return node

# 生成霍夫曼编码
def huffman(nodes):
    # 信源按照概率从小到大排序
    nodes = sort_nodes(nodes)
    # 重复合并两个信源，直至合并到根节点
    while len(nodes) > 1:
        node1 = nodes.pop(0)
        node2 = nodes.pop(0)
        node = merge_nodes(node1, node2)
        for i in range(len(nodes)):
            if nodes[i].prob > node.prob:
                nodes.insert(i, node)
                break
        else:
            nodes.append(node)
    # 从根节点开始，左子树编码为0，右子树编码为1
    codes = {}
    root = nodes[0]
    root.code = ''
    stack = [root]
    while stack:
        node = stack.pop()
        if node.symbol:
            codes[node.symbol] = node.code
        if node.left:
            node.left.code = node.code + '0'
            stack.append(node.left)
        if node.right:
            node.right.code = node.code + '1'
            stack.append(node.right)
    # 输出编码
    return codes

In [2]:
labels = [str(i) for i in range(8)]
prob = [45, 35, 20, 15, 15, 10, 5, 5]
nodes = [Node(label, p) for label, p in zip(labels, prob)]
codes = huffman(nodes)
codes

{'0': '11',
 '7': '10111',
 '6': '10110',
 '5': '1010',
 '2': '100',
 '1': '01',
 '4': '001',
 '3': '000'}